# Playground Series S6E4 - Predicting Irrigation Need EDA

This notebook develops an evidence-based understanding of the irrigation-need dataset before modeling. The analysis checks data quality, target balance, feature behavior, train/test alignment, and the strongest patterns to carry into baseline experiments.

## 1. Setup and Data Loading

Prepare the notebook environment, locate the Kaggle competition files, and load train, test, and sample submission data.

### 1.1 Environment

Load the standard Kaggle Python stack and define shared constants used throughout the analysis.

In [ ]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

sns.set_theme(style="whitegrid", context="notebook")
VIRIDIS_CMAP = "viridis"
VIRIDIS_COLORS = sns.color_palette(VIRIDIS_CMAP, n_colors=8)
TARGET_ORDER = ["Low", "Medium", "High"]
TARGET_PALETTE = dict(
    zip(TARGET_ORDER, sns.color_palette(VIRIDIS_CMAP, n_colors=len(TARGET_ORDER)))
)
RANDOM_STATE = 42

### 1.2 Load Competition Files

Discover the mounted Kaggle input files and infer the ID and target columns from `sample_submission.csv`.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

if not INPUT_ROOT.exists():
    raise RuntimeError(
        "This notebook is intended to run on Kaggle, where /kaggle/input is available."
    )


def find_file(filename: str) -> Path:
    """Return the first Kaggle input file matching the requested name.

    Args:
        filename: File name to search for below `/kaggle/input`.

    Returns:
        Path to the selected input file.
    """
    matches = sorted(INPUT_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under {INPUT_ROOT}. "
            "Attach the competition data to this notebook."
        )
    if len(matches) > 1:
        print(f"Found multiple {filename} files. Using: {matches[0]}")
        for match in matches:
            print(" -", match)
    return matches[0]


train_path = find_file("train.csv")
test_path = find_file("test.csv")
sample_submission_path = find_file("sample_submission.csv")

print("train:", train_path)
print("test:", test_path)
print("sample submission:", sample_submission_path)

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

target_col = sample_submission.columns[-1]
id_col = sample_submission.columns[0]

print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")
print(f"sample submission shape: {sample_submission.shape}")
print(f"id column: {id_col}")
print(f"target column: {target_col}")

## 2. Dataset Structure and Quality

Review the visible schema, feature types, missing values, and duplicate IDs to confirm whether cleaning is required before analysis.

### 2.1 Preview Records

Display the first few rows from train, test, and sample submission to confirm the visible schema and submission format.

In [ ]:
display(train.head())
display(test.head())
display(sample_submission.head())

### 2.2 Schema, Cardinality, and Duplicates

Summarize data types, missing counts, feature cardinality, and duplicate IDs.

In [ ]:
overview = pd.DataFrame(
    {
        "train_dtype": train.dtypes.astype(str),
        "test_dtype": test.dtypes.astype(str).reindex(train.columns),
        "train_missing": train.isna().sum(),
        "train_missing_pct": train.isna().mean() * 100,
        "train_unique": train.nunique(dropna=False),
    }
)
display(
    overview.sort_values(["train_missing_pct", "train_unique"], ascending=[False, True])
)

In [ ]:
feature_cols = [c for c in train.columns if c not in {id_col, target_col}]
numeric_cols = train[feature_cols].select_dtypes(include=np.number).columns.tolist()
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

print(f"Number of features: {len(feature_cols)}")
print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical / object features: {len(categorical_cols)}")
print(f"Duplicate rows in train: {train.duplicated().sum()}")
train_duplicate_ids = (
    train[id_col].duplicated().sum() if id_col in train else "id not found"
)
test_duplicate_ids = (
    test[id_col].duplicated().sum() if id_col in test else "id not found"
)

print(f"Duplicate ids in train: {train_duplicate_ids}")
print(f"Duplicate ids in test: {test_duplicate_ids}")

## 3. Target and Feature Distributions

Assess the target balance and the main numeric feature distributions. These checks determine how validation and model diagnostics should be designed.

In [ ]:
target_counts = train[target_col].value_counts(dropna=False)
target_pct = train[target_col].value_counts(normalize=True, dropna=False).mul(100)
target_summary = pd.concat(
    [target_counts.rename("count"), target_pct.rename("pct")], axis=1
)
display(target_summary)

plot_target_order = [x for x in TARGET_ORDER if x in target_counts.index]
plot_target_order += [x for x in target_counts.index if x not in plot_target_order]

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(
    data=train,
    x=target_col,
    order=plot_target_order,
    hue=target_col,
    hue_order=plot_target_order,
    palette=TARGET_PALETTE,
    legend=False,
    ax=ax,
)
ax.set_title("Target Distribution")
ax.set_xlabel(target_col)
ax.set_ylabel("Rows")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

### 3.1 Missing Values

Verify whether train and test contain missing feature values and whether missingness differs between the two datasets.

In [ ]:
missing = pd.DataFrame(
    {
        "train_missing_pct": train[feature_cols].isna().mean().mul(100),
        "test_missing_pct": test[[c for c in feature_cols if c in test.columns]]
        .isna()
        .mean()
        .mul(100),
    }
).fillna(0)
missing["abs_diff_pct"] = (
    missing["train_missing_pct"] - missing["test_missing_pct"]
).abs()

display(
    missing.sort_values(["train_missing_pct", "abs_diff_pct"], ascending=False).head(30)
)

plot_missing = missing[
    (missing["train_missing_pct"] > 0) | (missing["test_missing_pct"] > 0)
].sort_values("train_missing_pct", ascending=False)
if plot_missing.empty:
    print("No missing values found in train/test features.")
else:
    ax = (
        plot_missing[["train_missing_pct", "test_missing_pct"]]
        .head(40)
        .plot(kind="bar", figsize=(12, 5))
    )
    ax.set_title("Missing Values by Feature")
    ax.set_ylabel("Missing %")
    ax.set_xlabel("Feature")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

### 3.2 Numeric Feature Summary

Review numeric ranges, distributions, and correlations to identify continuous drivers and possible redundancy.

#### 3.2.1 Summary Statistics

Review central tendency, spread, and observed ranges for each numeric feature.

In [ ]:
if numeric_cols:
    display(train[numeric_cols].describe().T)
else:
    print("No numeric features detected.")

#### 3.2.2 Distributions and Correlations

Visualize numeric distributions and pairwise correlations to understand feature shape and redundancy.

In [ ]:
def plot_numeric_histograms(
    df: pd.DataFrame, cols: list[str], max_cols: int = 16
) -> None:
    """Plot histograms for the selected numeric columns.

    Args:
        df: Source data frame.
        cols: Numeric columns to plot.
        max_cols: Maximum number of columns to include.
    """
    cols = cols[:max_cols]
    if not cols:
        print("No numeric columns to plot.")
        return
    ncols = 4
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.2 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, cols):
        sns.histplot(df[col], bins=40, kde=False, color=VIRIDIS_COLORS[4], ax=ax)
        ax.set_title(col)
        ax.set_xlabel("")
    for ax in axes[len(cols) :]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


plot_numeric_histograms(train, numeric_cols, max_cols=24)

In [ ]:
if numeric_cols:
    corr = train[numeric_cols].corr(numeric_only=True)
    mask = np.triu(np.ones_like(corr, dtype=bool))
    plt.figure(
        figsize=(
            min(18, 1 + 0.45 * len(numeric_cols)),
            min(14, 1 + 0.45 * len(numeric_cols)),
        )
    )
    sns.heatmap(
        corr, mask=mask, cmap=VIRIDIS_CMAP, linewidths=0.3, cbar_kws={"shrink": 0.8}
    )
    plt.title("Numeric Feature Correlations")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric features detected.")

## 4. Target Relationships and Signal

Compare feature behavior across `Low`, `Medium`, and `High`, then rank individual feature signal.

### 4.1 Numeric Separation

Boxplots show how numeric feature distributions shift across target classes.

In [ ]:
if numeric_cols:
    top_numeric = numeric_cols[:12]
    ncols = 3
    nrows = int(np.ceil(len(top_numeric) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, top_numeric):
        sns.boxplot(
            data=train,
            x=target_col,
            y=col,
            order=plot_target_order,
            hue=target_col,
            hue_order=plot_target_order,
            palette=TARGET_PALETTE,
            legend=False,
            ax=ax,
            showfliers=False,
        )
        ax.set_title(col)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=20)
    for ax in axes[len(top_numeric) :]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric features detected.")

### 4.2 Categorical Separation

Normalized categorical target mixes highlight groups with elevated `Medium` or `High` irrigation need.

In [ ]:
if categorical_cols:
    for col in categorical_cols[:8]:
        ct = pd.crosstab(train[col], train[target_col], normalize="index").sort_index()
        display(ct.head(30))
        ax = ct.head(30).plot(
            kind="bar", stacked=True, figsize=(12, 4), colormap=VIRIDIS_CMAP
        )
        ax.set_title(f"Target Mix by {col}")
        ax.set_ylabel("Share within category")
        ax.set_xlabel(col)
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
else:
    print("No categorical features detected.")

### 4.3 Univariate Signal Ranking

Mutual information provides a quick model-agnostic view of individual feature signal.

In [ ]:
try:
    from sklearn.feature_selection import mutual_info_classif
    from sklearn.preprocessing import OrdinalEncoder

    X = train[feature_cols].copy()
    y = train[target_col]

    cat_cols_for_mi = X.select_dtypes(exclude=np.number).columns.tolist()
    num_cols_for_mi = X.select_dtypes(include=np.number).columns.tolist()

    for col in num_cols_for_mi:
        X[col] = X[col].fillna(X[col].median())

    if cat_cols_for_mi:
        X[cat_cols_for_mi] = X[cat_cols_for_mi].astype("string").fillna("__missing__")
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X[cat_cols_for_mi] = encoder.fit_transform(X[cat_cols_for_mi])

    discrete_mask = [col in cat_cols_for_mi for col in X.columns]
    mi = mutual_info_classif(
        X, y, discrete_features=discrete_mask, random_state=RANDOM_STATE
    )
    mi_df = pd.DataFrame({"feature": X.columns, "mutual_info": mi}).sort_values(
        "mutual_info", ascending=False
    )
    display(mi_df.head(30))

    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=mi_df.head(25),
        x="mutual_info",
        y="feature",
        hue="feature",
        palette=VIRIDIS_CMAP,
        legend=False,
    )
    plt.title("Top Features by Mutual Information")
    plt.xlabel("Mutual information")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Skipped mutual information ranking:", repr(exc))

## 5. Train/Test Alignment

Compare train and test distributions. Low drift supports standard stratified validation.

### 5.1 Numeric Drift

Compare train and test numeric means using standardized mean differences.

In [ ]:
common_numeric = [c for c in numeric_cols if c in test.columns]

drift_rows = []
for col in common_numeric:
    train_mean = train[col].mean()
    test_mean = test[col].mean()
    train_std = train[col].std()
    test_std = test[col].std()
    pooled_std = pd.concat([train[col], test[col]], axis=0).std()
    standardized_mean_diff = (
        np.nan if pooled_std == 0 else abs(train_mean - test_mean) / pooled_std
    )
    drift_rows.append(
        {
            "feature": col,
            "train_mean": train_mean,
            "test_mean": test_mean,
            "train_std": train_std,
            "test_std": test_std,
            "standardized_mean_diff": standardized_mean_diff,
        }
    )

drift = pd.DataFrame(drift_rows).sort_values("standardized_mean_diff", ascending=False)
display(drift.head(30))

### 5.2 Distribution Overlay

Overlay train and test distributions for the features with the largest measured differences.

In [ ]:
if common_numeric:
    plot_cols = drift["feature"].head(12).tolist()
    ncols = 3
    nrows = int(np.ceil(len(plot_cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.6 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, plot_cols):
        sns.kdeplot(
            train[col], label="train", ax=ax, fill=False, color=VIRIDIS_COLORS[2]
        )
        sns.kdeplot(test[col], label="test", ax=ax, fill=False, color=VIRIDIS_COLORS[6])
        ax.set_title(col)
        ax.set_xlabel("")
        ax.legend()
    for ax in axes[len(plot_cols) :]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No common numeric features detected.")

## 6. Deep-Dive Diagnostics

Translate first-pass observations into model-oriented diagnostics: ordered target movement, categorical lift, interactions, and threshold behavior.

### 6.1 Ordered Target Movement

Treat `Low`, `Medium`, and `High` as an ordered scale to inspect monotonic numeric movement.

In [ ]:
if set(TARGET_ORDER).issubset(set(train[target_col].dropna().unique())):
    target_rank = {label: idx for idx, label in enumerate(TARGET_ORDER)}
else:
    target_rank = {
        label: idx
        for idx, label in enumerate(sorted(train[target_col].dropna().unique()))
    }

y_ranked = train[target_col].map(target_rank)

if numeric_cols:
    target_numeric_means = (
        train.groupby(target_col)[numeric_cols].mean().reindex(plot_target_order)
    )
    display(target_numeric_means.T)

    numeric_rank_corr = (
        train[numeric_cols]
        .corrwith(y_ranked, method="spearman")
        .rename("spearman_corr_with_target_rank")
        .to_frame()
    )
    numeric_rank_corr["abs_corr"] = numeric_rank_corr[
        "spearman_corr_with_target_rank"
    ].abs()
    numeric_rank_corr = numeric_rank_corr.sort_values("abs_corr", ascending=False)
    display(numeric_rank_corr)

    plt.figure(figsize=(8, 5))
    sns.barplot(
        data=numeric_rank_corr.reset_index().rename(columns={"index": "feature"}),
        x="spearman_corr_with_target_rank",
        y="feature",
        hue="feature",
        palette=VIRIDIS_CMAP,
        legend=False,
    )
    plt.axvline(0, color="black", linewidth=0.8)
    plt.title("Numeric Features vs Ordered Target Rank")
    plt.xlabel("Spearman correlation")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric features detected.")

### 6.2 Categorical Lift and Interactions

Measure which categories and category combinations lift the rare `High` class above its baseline rate.

In [ ]:
base_target_rate = train[target_col].value_counts(normalize=True)
cat_lift_rows = []

for col in categorical_cols:
    stats = (
        train.groupby(col)[target_col]
        .value_counts(normalize=True)
        .rename("rate")
        .reset_index()
    )
    counts = train[col].value_counts().rename("row_count")
    pivot = stats.pivot(index=col, columns=target_col, values="rate").fillna(0)
    pivot["row_count"] = counts.reindex(pivot.index)
    pivot["feature"] = col
    for label in plot_target_order:
        if label in pivot.columns and label in base_target_rate:
            pivot[f"{label}_lift"] = pivot[label] / base_target_rate[label]
    cat_lift_rows.append(pivot.reset_index().rename(columns={col: "category"}))

cat_lift = pd.concat(cat_lift_rows, ignore_index=True)
high_lift_col = "High_lift" if "High_lift" in cat_lift.columns else None

if high_lift_col:
    display(cat_lift.sort_values(high_lift_col, ascending=False).head(25))
else:
    display(cat_lift.head(25))

In [ ]:
interaction_pairs = [
    ("Crop_Growth_Stage", "Mulching_Used"),
    ("Crop_Growth_Stage", "Irrigation_Type"),
    ("Crop_Growth_Stage", "Water_Source"),
]

for left, right in interaction_pairs:
    if (
        left in train.columns
        and right in train.columns
        and "High" in train[target_col].unique()
    ):
        high_rate = train.assign(is_high=(train[target_col] == "High")).pivot_table(
            index=left,
            columns=right,
            values="is_high",
            aggfunc="mean",
        )
        display(high_rate)
        plt.figure(figsize=(9, 4.5))
        sns.heatmap(
            high_rate,
            annot=True,
            fmt=".3f",
            cmap=VIRIDIS_CMAP,
            cbar_kws={"label": "High rate"},
        )
        plt.title(f"High Irrigation Need Rate: {left} x {right}")
        plt.xlabel(right)
        plt.ylabel(left)
        plt.tight_layout()
        plt.show()

In [ ]:
cat_drift_rows = []
for col in categorical_cols:
    train_dist = train[col].value_counts(normalize=True, dropna=False)
    test_dist = test[col].value_counts(normalize=True, dropna=False)
    categories = sorted(
        set(train_dist.index).union(set(test_dist.index)), key=lambda x: str(x)
    )
    total_variation = 0.5 * sum(
        abs(train_dist.get(cat, 0) - test_dist.get(cat, 0)) for cat in categories
    )
    max_abs_diff = max(
        abs(train_dist.get(cat, 0) - test_dist.get(cat, 0)) for cat in categories
    )
    cat_drift_rows.append(
        {
            "feature": col,
            "total_variation_distance": total_variation,
            "max_category_abs_diff": max_abs_diff,
            "train_unique": train[col].nunique(dropna=False),
            "test_unique": test[col].nunique(dropna=False),
        }
    )

cat_drift = pd.DataFrame(cat_drift_rows).sort_values(
    "total_variation_distance", ascending=False
)
display(cat_drift)

plt.figure(figsize=(8, 4))
sns.barplot(
    data=cat_drift,
    x="total_variation_distance",
    y="feature",
    hue="feature",
    palette=VIRIDIS_CMAP,
    legend=False,
)
plt.title("Categorical Train/Test Drift")
plt.xlabel("Total variation distance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

### 6.3 Binned Numeric Target Rates

Binning the strongest numeric predictors makes threshold behavior easier to inspect.

In [ ]:
top_numeric_for_bins = [
    col
    for col in [
        "Soil_Moisture",
        "Rainfall_mm",
        "Temperature_C",
        "Wind_Speed_kmh",
        "Humidity",
        "Previous_Irrigation_mm",
    ]
    if col in train.columns
]

binned_rate_frames = []
for col in top_numeric_for_bins:
    binned = pd.qcut(train[col], q=10, duplicates="drop")
    rate_table = pd.crosstab(binned, train[target_col], normalize="index").reindex(
        columns=plot_target_order
    )
    rate_table["row_count"] = train.groupby(binned, observed=True).size()
    rate_table["feature"] = col
    binned_rate_frames.append(rate_table.reset_index().rename(columns={col: "bin"}))

binned_target_rates = pd.concat(binned_rate_frames, ignore_index=True)
display(binned_target_rates.head(60))

for col in top_numeric_for_bins[:4]:
    plot_df = binned_target_rates[binned_target_rates["feature"] == col].copy()
    plot_df["bin_label"] = plot_df["bin"].astype(str)
    available_targets = [
        label for label in plot_target_order if label in plot_df.columns
    ]
    ax = plot_df.set_index("bin_label")[available_targets].plot(
        kind="bar",
        stacked=True,
        figsize=(12, 4),
        colormap=VIRIDIS_CMAP,
    )
    ax.set_title(f"Target Rate by {col} Decile")
    ax.set_xlabel(f"{col} decile")
    ax.set_ylabel("Class share")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 7. Summary and Experiment Readiness

The EDA has answered the main pre-experiment questions: the data is clean, train/test drift is low, the target is imbalanced, and several strong feature patterns are clear.

Key insights from the latest Kaggle run:

- The dataset has `630,000` training rows, `270,000` test rows, `19` predictors, no missing values, no duplicate rows, and no duplicate IDs.
- The target is imbalanced: `Low` is `58.72%`, `Medium` is `37.95%`, and `High` is only `3.33%`. Validation should be stratified and class-level metrics are required.
- Train/test alignment is strong. Numeric standardized mean differences are tiny, with the largest around `0.004`; categorical total variation distances are also small, with the largest around `0.0025`.
- The strongest individual predictors are `Soil_Moisture`, `Rainfall_mm`, `Crop_Growth_Stage`, `Temperature_C`, `Wind_Speed_kmh`, `Previous_Irrigation_mm`, `Humidity`, and `Mulching_Used`.
- Average `Soil_Moisture` falls from `43.31` for `Low` to `17.67` for `High`, while `Temperature_C` rises from `25.35` to `34.57`, and `Wind_Speed_kmh` rises from `9.22` to `14.64`.
- `Flowering` and `Vegetative` have roughly `1.93x` the baseline `High` rate. Fields without mulching have a `High` rate of `5.85%`, compared with `0.79%` for fields with mulching.
- The strongest interactions are practical and modelable: `Vegetative + No mulching` has about `10.81%` `High`, `Flowering + No mulching` has about `10.61%` `High`, and `Flowering/Vegetative + River` are both around `8.3%` `High`.
- Binned target rates reveal threshold behavior: the lowest soil-moisture deciles have `High` rates around `9.6%` to `11.7%`, and the lowest rainfall decile has a `High` rate around `14.6%`.

Recommended experiment design:

- Use stratified validation.
- Report accuracy plus macro F1, balanced accuracy, log loss where probabilities are available, and confusion matrices.
- Prioritize CatBoost after the baseline run, but keep simpler models as sanity checks.
- Test targeted interaction features from EDA: `Crop_Growth_Stage x Mulching_Used`, `Crop_Growth_Stage x Water_Source`, and `Crop_Growth_Stage x Irrigation_Type`.
- Monitor precision and recall for `High`, because it is rare and likely operationally important.

## 8. EDA Notes

Use this section while running on Kaggle to capture additional observations that should influence feature engineering, validation design, or model diagnostics.